In [1]:
import json
import re
import pandas as pd
from datasets import Dataset
from ragas import evaluate
# Altere esta linha abaixo:
from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextPrecisionWithoutReference
import logging
import sys
import os
from getpass import getpass
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import time
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import llm_factory
from openai import OpenAI

/tmp/ipykernel_154503/169786579.py:7: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextPrecisionWithoutReference
/tmp/ipykernel_154503/169786579.py:7: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, LLMContextPrecisionWithoutReference
/tmp/ipykernel_154503/169786579.py:7: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference

In [2]:
# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Gemini API Key: ")
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Hugging Face Token: ")
# if "GROQ_API_KEY" not in os.environ:
#     os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Digite sua OpenAI API Key: ")

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo detectado: {device}")
model_name = "tcepi/sts_bertimbau"

# Configurações para otimizar o uso da memória de vídeo (VRAM)
model_kwargs = { 'device': device }
encode_kwargs = {'normalize_embeddings': True, 'batch_size': 32} # Batch menor para evitar estouro de memória na MX350

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
    multi_process=False
)

#llm_avaliador = init_chat_model(model="google_genai:gemini-3.1-flash-lite", temperature=0.0)

#llm_avaliador = init_chat_model(model="openai:gpt-4o-mini", temperature=0.0)
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
llm_ragas = llm_factory("gpt-5-mini", client=openai_client, max_tokens=8192, temperature=0.0) #gpt-5-mini-2025-08-07


Dispositivo detectado: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [16]:
def criar_dataframe(caminho_arquivo):
    # 1. Leitura nativa do arquivo JSON
    with open(caminho_arquivo, "r", encoding="utf-8") as f:
        dados_json = json.load(f)

    lista_para_dataframe = []

    # 2. Processamento de cada elemento da lista
    for item in dados_json:
        # Extração e limpeza do campo 'documentos_retornados_rag'
        docs_originais = item.get("documentos_retornados_rag", {})
        docs_limpos = []

        # Extrai apenas os valores (.values()) do objeto/dicionário
        for texto in docs_originais.values():
            # Regex: '^.*?' vai do início até encontrar "ANO:" de forma "lazy" (preguiçosa).
            # Substituímos tudo isso por "ANO:", eliminando o que estava antes.
            texto_processado = re.sub(r"^.*?ANO:", "ANO:", texto)
            docs_limpos.append(texto_processado)

        # Montagem do dicionário estruturado para a linha do DataFrame
        
        linha = {
            "id": str(item.get("id")),
            "input_usuario": str(item.get("input_usuario")),
            "demanda": str(item.get("demanda_estruturada").get("demanda")),
            "pergunta_critica_1": str(item.get("demanda_estruturada").get("pergunta_critica_1")),
            "pergunta_critica_2": str(item.get("demanda_estruturada").get("pergunta_critica_2")),
            "documentos_retornados_rag": docs_limpos,  # Lista de strings limpas
            "resposta_gerada": str(item.get("resposta_gerada")),
            "veredito_validador": str(item.get("veredito_validador")),
            "fonte_verificada": item.get("fonte_verificada"),
            "referencias": item.get("referencias")
        }

        lista_para_dataframe.append(linha)

    # 3. Criação do DataFrame final com as colunas estruturadas
    df_final = pd.DataFrame(lista_para_dataframe)

    return df_final

criar_dataframe('resultados_finais.json').to_pickle("resultados_finais.pkl")

In [4]:
df = pd.read_pickle("resultados_ragas2.pkl")

In [8]:
len(df)

30

### RAGAS

In [6]:
def ragas(df_resultados,i,j, embeddings_avaliador, caminho_pkl="resultados_ragas2.pkl"):
    # 1. Cria uma cópia para evitar side-effects destrutivos no DF original
    df_lote = df_resultados.iloc[i:j].copy()
    if df_lote.empty:
        print(f"⚠️ O intervalo [{i}:{j}] está vazio. Nada para processar.")
        return None
    
    embeddings_wrapper = LangchainEmbeddingsWrapper(embeddings_avaliador)
    
    questions_formatadas = []

    # Iteramos pelas colunas do DataFrame do lote para construir o parágrafo investigativo
    for demanda, p1, p2 in zip(
        df_lote["demanda"].tolist(), 
        df_lote["pergunta_critica_1"].tolist(), 
        df_lote["pergunta_critica_2"].tolist()
    ):
        # Base da string com a demanda e a pergunta crítica 1
        q_ragas = (
            f"{demanda} No que diz respeito a esse cenário, é fundamental averiguar: {p1}"
        )
        
        # Adiciona a pergunta crítica 2 com o conectivo apenas se ela existir e não for nula/vazia
        if p2 and len(str(p2).strip()) > 7:  # Evita strings muito curtas ou nulas
            q_ragas += f" Adicionalmente, torna-se necessário esclarecer: {p2}"
            
        questions_formatadas.append(q_ragas)

    print(f"🚀 Iniciando avaliação do lote: linhas {i} até {j}...")
    # 2. Prepara o dataset estruturado para o RAGAS
    dados_formatados = {
        "question": questions_formatadas,
        "contexts": df_lote["documentos_retornados_rag"].tolist(), 
        "answer": df_lote["resposta_gerada"].tolist()
    }
    dataset_hf = Dataset.from_dict(dados_formatados)
    
    logging.basicConfig(stream=sys.stdout, level=logging.INFO)
    # Define especificamente o escopo do RAGAS para ver os passos internos
    logging.getLogger("ragas").setLevel(logging.INFO)
    faithfulness = Faithfulness(llm=llm_ragas)
    answer_relevancy = AnswerRelevancy(llm=llm_ragas)
    context_precision = LLMContextPrecisionWithoutReference(llm=llm_ragas)
    metricas_para_avaliar = [
        ("faithfulness", faithfulness),
        ("answer_relevancy", answer_relevancy),
        ("context_precision", context_precision),
    ]

    for nome_metrica, objeto_metrica in metricas_para_avaliar:
        print(f"Avaliando a métrica: {nome_metrica}")
        resultado = evaluate(
            dataset=dataset_hf,
            metrics=[objeto_metrica],  # Uma por vez!
            embeddings=embeddings_wrapper,
            raise_exceptions=False,
        )
        df_detalhado = resultado.to_pandas()
        nome_real_no_ragas = objeto_metrica.name
        print(f'Nome real da métrica no RAGAS: {nome_real_no_ragas}')
        try:
            # O RAGAS costuma gerar colunas extras com os textos intermediários
            # Vamos salvar todas as colunas daquela linha para análise visual posterior
            with open("raciocinios_ragas2.txt", "a", encoding="utf-8") as f:
                f.write(f"=== ÍNDICE DO DATAFRAME: {i} | MÉTRICA: {nome_metrica} ===\n")
                f.write(f"Pergunta: {df_lote['input_usuario'].values[0]}\n")
                f.write(f"Nota Atribuída: {df_detalhado[nome_real_no_ragas].values[0]}\n")
                f.write(f"Dados Completos do Julgamento:\n")
                f.write(f"{df_detalhado.to_string(index=False)}\n")
                f.write("="*60 + "\n\n")
        except Exception as e:
            print(f"⚠️ Não foi possível extrair o raciocínio textual desta linha: {e}")
        print(f"Resultado da métrica '{nome_metrica}': {resultado.to_pandas()[nome_real_no_ragas].values}")
        df_lote[nome_metrica] = df_detalhado[nome_real_no_ragas].values
        time.sleep(60)         
    
    print("✅ Métricas avaliadas e integradas ao DataFrame original com sucesso!")
    if os.path.exists(caminho_pkl):
        print(f"📂 Arquivo '{caminho_pkl}' encontrado. Carregando histórico...")
        df_existente = pd.read_pickle(caminho_pkl)
        df_final = pd.concat([df_existente, df_lote], ignore_index=True) 
    else:
        print(f"✨ Arquivo '{caminho_pkl}' não existe. Criando um novo.")
        df_final = df_lote

    # 6. Salva o DataFrame consolidado de volta em .pkl
    df_final.to_pickle(caminho_pkl)

In [ ]:
total_linhas = len(df)

# Loop externo que envia uma linha por vez para a função
for linha_atual in range(5, total_linhas):
    prox_linha = linha_atual + 1
    
    if "INADEQUADO" in df["veredito_validador"].iloc[linha_atual]:
        print(f"⏩ Linha {linha_atual} pulada: veredito é 'INADEQUADO'.")
        continue
    
    # Executa a avaliação apenas para o caso atual [0:1], depois [1:2], etc.
    ragas(
        df_resultados=df, 
        i=linha_atual, 
        j=prox_linha, 
        embeddings_avaliador=embeddings
    )
    
    print(f"☕ Pausa de segurança após a linha {linha_atual}...")
    time.sleep(5)  # Pequena pausa opcional para estabilizar a rede entre linhas

In [21]:
def ragas_baseline(df_resultados,respostas,i,j, embeddings_avaliador, caminho_pkl="resultados_inadequado.pkl"):
    # 1. Cria uma cópia para evitar side-effects destrutivos no DF original
    df_lote = df_resultados.iloc[i:j].copy()
    if df_lote.empty:
        print(f"⚠️ O intervalo [{i}:{j}] está vazio. Nada para processar.")
        return None
    
    embeddings_wrapper = LangchainEmbeddingsWrapper(embeddings_avaliador)
    
    questions_formatadas = []

    # Iteramos pelas colunas do DataFrame do lote para construir o parágrafo investigativo
    for demanda, p1, p2 in zip(
        df_lote["demanda"].tolist(), 
        df_lote["pergunta_critica_1"].tolist(), 
        df_lote["pergunta_critica_2"].tolist()
    ):
        # Base da string com a demanda e a pergunta crítica 1
        q_ragas = (
            f"{demanda} No que diz respeito a esse cenário, é fundamental averiguar: {p1}"
        )
        
        # Adiciona a pergunta crítica 2 com o conectivo apenas se ela existir e não for nula/vazia
        if p2 and len(str(p2).strip()) > 7:  # Evita strings muito curtas ou nulas
            q_ragas += f" Adicionalmente, torna-se necessário esclarecer: {p2}"
            
        questions_formatadas.append(q_ragas)

    print(f"🚀 Iniciando avaliação do lote: linhas {i} até {j}...")
    # 2. Prepara o dataset estruturado para o RAGAS
    dados_formatados = {
        "question": questions_formatadas,
        "answer": [respostas[i]["resposta"]]
    }
    dataset_hf = Dataset.from_dict(dados_formatados)
    
    df_lote["resposta_baseline"] = respostas[i]["resposta"]
    
    logging.basicConfig(stream=sys.stdout, level=logging.INFO)
    # Define especificamente o escopo do RAGAS para ver os passos internos
    logging.getLogger("ragas").setLevel(logging.INFO)
    answer_relevancy = AnswerRelevancy(llm=llm_ragas)
    metricas_para_avaliar = [
        ("answer_relevancy", answer_relevancy),
    ]

    for nome_metrica, objeto_metrica in metricas_para_avaliar:
        print(f"Avaliando a métrica: {nome_metrica}")
        resultado = evaluate(
            dataset=dataset_hf,
            metrics=[objeto_metrica],  # Uma por vez!
            embeddings=embeddings_wrapper,
            raise_exceptions=False,
        )
        df_detalhado = resultado.to_pandas()
        nome_real_no_ragas = objeto_metrica.name
        print(f'Nome real da métrica no RAGAS: {nome_real_no_ragas}')
       
        print(f"Resultado da métrica '{nome_metrica}': {resultado.to_pandas()[nome_real_no_ragas].values}")
        df_lote[nome_metrica+"_baseline"] = df_detalhado[nome_real_no_ragas].values
        time.sleep(15)         
    
    print("✅ Métricas avaliadas e integradas ao DataFrame original com sucesso!")
    if os.path.exists(caminho_pkl):
        print(f"📂 Arquivo '{caminho_pkl}' encontrado. Carregando histórico...")
        df_existente = pd.read_pickle(caminho_pkl)
        df_final = pd.concat([df_existente, df_lote], ignore_index=True) 
    else:
        print(f"✨ Arquivo '{caminho_pkl}' não existe. Criando um novo.")
        df_final = df_lote

    # 6. Salva o DataFrame consolidado de volta em .pkl
    df_final.to_pickle(caminho_pkl)

In [ ]:
df_existente = pd.read_pickle('resultados_finais.pkl')

caminho_arquivo_entrada = "resultados_baseline.json"
with open(caminho_arquivo_entrada, 'r', encoding='utf-8') as f:
    respostas = json.load(f)
    print(f"📄 Arquivo '{caminho_arquivo_entrada}' carregado com sucesso. Total de respostas: {len(respostas)}")
# Loop externo que envia uma linha por vez para a função
for linha_atual in range(0, len(df_existente)):
    prox_linha = linha_atual + 1
    
    if "INADEQUADO" not in df_existente["veredito_validador"].iloc[linha_atual]:
        print(f"⏩ Linha {linha_atual} pulada: veredito é 'ADEQUADO'.")
        continue
    
    # Executa a avaliação apenas para o caso atual [0:1], depois [1:2], etc.
    ragas_baseline(
        df_resultados=df_existente, 
        respostas=respostas, 
        i=linha_atual, 
        j=prox_linha, 
        embeddings_avaliador=embeddings
    )
    
    print(f"☕ Pausa de segurança após a linha {linha_atual}...")
    time.sleep(5)  # Pequena pausa opcional para estabilizar a rede entre linhas

In [ ]:
df_existente = pd.read_pickle('resultados_ragas3.pkl')
df_existente

In [29]:
df2 = pd.read_pickle('resultados_inadequado.pkl')
df2.head()

,id,input_usuario,demanda,pergunta_critica_1,pergunta_critica_2,documentos_retornados_rag,resposta_gerada,veredito_validador,fonte_verificada,referencias,resposta_baseline,answer_relevancy_baseline
0,3,O alumínio é liberado na comida quando aquecid...,Qual é o risco real à saúde humana associado à...,Existe evidência científica que comprove que a...,Quais são as diretrizes de órgãos de saúde e s...,[],,INADEQUADO,False,[],"Essa é uma questão que envolve química, saúde ...",0.796293
1,6,A cidade de SP que teve água contaminada com m...,Quais são os procedimentos oficiais para verif...,Existe algum registro oficial ou comunicado da...,None,[],,INADEQUADO,False,[],É importante esclarecer um ponto fundamental a...,0.819324
2,7,É verdade que arroz vira açúcar no corpo? Mas ...,Qual é o impacto do consumo de arroz nos nívei...,Como o organismo processa o amido presente no ...,None,[],,INADEQUADO,False,[],"A resposta curta é **sim, o arroz vira açúcar ...",0.511636
3,10,Conheço várias pessoas que fizeram quimio e mo...,Quais são os benefícios e os objetivos clínico...,Quais são os objetivos terapêuticos da quimiot...,Como a literatura científica e os protocolos o...,[],,INADEQUADO,False,[],Sinto muito que você tenha passado por experiê...,0.679872
4,12,"Meu Deus, olha isso! Vocês viram que foram enc...",Quais são os protocolos de segurança alimentar...,Existe alguma evidência científica ou relatóri...,A alegação sobre a presença de DNA humano em c...,[],,INADEQUADO,False,[],Essa história que você trouxe circula na inter...,0.773496


In [33]:
df1 = pd.read_pickle('resultados_ragas3.pkl')
df2 = pd.read_pickle('resultados_inadequado.pkl')
df_final = pd.concat([df1, df2], ignore_index=True)
df_final = df_final.sort_values(by='id', ascending=True, key=lambda x: x.astype(int))
df_final = df_final.reset_index(drop=True)
df_final.to_pickle("resultados_ragas.pkl")

In [35]:
df_final = pd.read_pickle("resultados_ragas.pkl")

id_numerico = pd.to_numeric(df_final['id'])

# 2. Aplica a lógica matemática de grupos de 10 em 10
# O operador '//' faz a divisão inteira (descarta as casas decimais)
df_final['grupo'] = ((id_numerico - 1) // 10 + 1)

# 3. Converte a nova coluna 'grupo' para string (para manter o padrão do seu projeto)
df_final['grupo'] = df_final['grupo'].astype(str)

df_final.to_pickle("resultados_ragas.pkl")

In [40]:
df_final = pd.read_pickle("resultados_ragas.pkl")
df_final

,id,input_usuario,demanda,pergunta_critica_1,pergunta_critica_2,documentos_retornados_rag,resposta_gerada,veredito_validador,fonte_verificada,referencias,faithfulness,answer_relevancy,context_precision,resposta_baseline,answer_relevancy_baseline,grupo
0,1,É verdade que paracetamol causa autismo? Se o ...,Qual é o consenso científico atual sobre a seg...,Existe evidência científica robusta que estabe...,A afirmação feita pela figura pública citada é...,[ANO: 2025 | FONTE_VERIFICADA: True | FONTE: a...,O paracetamol continua sendo considerado o ana...,ADEQUADO,True,[https://www.agencialupa.org/jornalismo/2025/0...,1.000000,0.731821,1.0,A resposta curta é **não**. Não existe nenhuma...,0.628508,1
1,2,O detergente Ypê estava contaminado por bactér...,Quais são os protocolos de segurança e os proc...,Existe uma relação de causalidade entre a regu...,Qual é o respaldo oficial sobre o caso de cont...,[ANO: 2024 | FONTE_VERIFICADA: True | FONTE: a...,A segurança sanitária no Brasil é garantida po...,ADEQUADO,True,[https://www.aosfatos.org/noticias/falso-anvis...,0.583333,0.797245,1.0,A sua preocupação toca em um ponto fundamental...,0.918824,1
2,3,O alumínio é liberado na comida quando aquecid...,Qual é o risco real à saúde humana associado à...,Existe evidência científica que comprove que a...,Quais são as diretrizes de órgãos de saúde e s...,[],,INADEQUADO,False,[],NaN,NaN,NaN,"Essa é uma questão que envolve química, saúde ...",0.796293,1
3,4,O governo suspendeu a nova vacina contra dengu...,Quais são as formas cientificamente comprovada...,Quais são os critérios técnicos e científicos ...,Existe uma relação de causalidade comprovada e...,[ANO: 2026 | FONTE_VERIFICADA: True | FONTE: g...,É perfeitamente compreensível que você sinta a...,ADEQUADO,True,[https://www.gov.br/saude/pt-br/assuntos/saude...,0.862069,0.812681,1.0,"Essa é uma questão que envolve química, saúde ...",0.674495,1
4,5,O nipah tá se espalhando no mundo. Fui no carn...,Quais são as formas de transmissão do vírus Ni...,Qual é a atual situação epidemiológica do víru...,Quais são as formas de transmissão confirmadas...,[ANO: 2026 | FONTE_VERIFICADA: True | FONTE: c...,É perfeitamente compreensível que você sinta a...,ADEQUADO,True,[https://www.aosfatos.org/noticias/o-que-e-vir...,0.680000,0.760476,1.0,É importante esclarecer alguns pontos sobre o ...,0.509158,1
5,6,A cidade de SP que teve água contaminada com m...,Quais são os procedimentos oficiais para verif...,Existe algum registro oficial ou comunicado da...,None,[],,INADEQUADO,False,[],NaN,NaN,NaN,É importante esclarecer um ponto fundamental a...,0.819324,1
6,7,É verdade que arroz vira açúcar no corpo? Mas ...,Qual é o impacto do consumo de arroz nos nívei...,Como o organismo processa o amido presente no ...,None,[],,INADEQUADO,False,[],NaN,NaN,NaN,"A resposta curta é **sim, o arroz vira açúcar ...",0.511636,1
7,8,Li uma postagem no Instagram sobre um estudo q...,Quais são os efeitos do consumo de vinho na mi...,Existe evidência científica robusta que compro...,O consumo de vinho é recomendado por associaçõ...,[ANO: 2026 | FONTE_VERIFICADA: True | FONTE: a...,A higiene bucal eficaz é alcançada exclusivame...,ADEQUADO,True,[https://www.agencialupa.org/verificacao/2026/...,0.545455,0.729209,1.0,"É compreensível que, após grandes eventos como...",0.656615,1
8,9,Me avisaram para não sair no Carnaval por caus...,O que é a 'pneumonia branca' e existe algum su...,Existe registro oficial ou científico de uma n...,None,[ANO: 2026 | FONTE_VERIFICADA: True | FONTE: a...,É perfeitamente compreensível que você se sint...,ADEQUADO,True,[https://www.aosfatos.org/noticias/nao-e-verda...,0.857143,0.852003,1.0,É importante esclarecer um ponto fundamental a...,0.679243,1
9,10,Conheço várias pessoas que fizeram quimio e mo...,Quais são os benefícios e os objetivos clínico...,Quais são os objetivos terapêuticos da quimiot...,Como a literatura científica e os protocolos o...,[],,INADEQUADO,False,[],NaN,NaN,NaN,Sinto muito que você tenh

### Veredito "ADEQUADO" por grupo

In [ ]:
df["id_numerico"] = pd.to_numeric(df["id"])

# 2. Define as regras dos intervalos (bins) e as etiquetas de texto (labels)
# O intervalo é aberto à esquerda e fechado à direita, ex: (0, 10] engloba de 1 a 10.
intervalos = [0, 10, 20, 30]
nomes_grupos = ["1-10", "11-20", "21-30"]

# 3. Cria uma nova coluna temporária categorizando cada ID em seu respectivo grupo
df["grupo_id"] = pd.cut(
    df["id_numerico"], bins=intervalos, labels=nomes_grupos
)

# 4. Filtra apenas onde o veredito é "ADEQUADO" e conta por grupo
resultado = (
    df[df["veredito_validador"] == "ADEQUADO"]
    .groupby("grupo_id", observed=False)
    .size()
)

# Exibe o resultado formatado
print("Quantidade de 'ADEQUADO' por grupo de ID:")
print(resultado)

Quantidade de 'ADEQUADO' por grupo de ID:
grupo_id
1-10     7
11-20    8
21-30    4
dtype: int64


In [12]:
condicao_inadequado = df["veredito_validador"] == "INADEQUADO"

# Condição 2: O tamanho da lista de documentos é igual a 0
condicao_lista_vazia = df["documentos_retornados_rag"].str.len() == 0

# Combinando ambas as condições com o operador operador '&' (E lógico) e somando os True
total_casos = (condicao_inadequado & condicao_lista_vazia).sum()

print(
    f"Quantidade de linhas INADEQUADO: {condicao_inadequado.sum()}\n"
    f"Quantidade de linhas INADEQUADO com 0 documentos retornados: {total_casos}"
)

Quantidade de linhas INADEQUADO: 11
Quantidade de linhas INADEQUADO com 0 documentos retornados: 11


In [11]:
# 1. Função auxiliar para contar palavras de cada string da lista e somar as extensões (lens)
def somar_palavras_da_lista(lista_textos):
    if not isinstance(lista_textos, list) or len(lista_textos) == 0:
        return 0
    
    # Faz o len().split() para cada texto individual e soma tudo
    return sum(len(str(texto).split()) for texto in lista_textos)


# 2. Aplica a função no DataFrame (ajustado para o nome da sua coluna)
df["soma_palavras_documentos"] = df["documentos_retornados_rag"].apply(somar_palavras_da_lista)

# 3. Calcula as estatísticas solicitadas
maximo_geral = df["soma_palavras_documentos"].max()
media_por_entrada = df["soma_palavras_documentos"].mean()

# 4. Exibe os resultados formatados
print("📊 --- ANÁLISE DETALHADA POR SOMA DE DOCUMENTOS ---")
print(f"• Máximo geral (pico de palavras em uma entrada): {maximo_geral:,}")
print(f"• Média de palavras por entrada na base:          {media_por_entrada:.2f}")

📊 --- ANÁLISE DETALHADA POR SOMA DE DOCUMENTOS ---
• Máximo geral (pico de palavras em uma entrada): 5,246
• Média de palavras por entrada na base:          1412.13


In [18]:
df.head()

,id,input_usuario,documentos_retornados_rag,resposta_gerada,veredito_validador,fonte_verificada,referencias
0,1,É verdade que paracetamol causa autismo? Se o ...,[ANO: 2025 | FONTE_VERIFICADA: True | FONTE: a...,O consenso científico atual estabelece que o p...,ADEQUADO,True,[https://www.agencialupa.org/jornalismo/2025/0...
1,2,O detergente Ypê estava contaminado por bactér...,[ANO: 2025 | FONTE_VERIFICADA: True | FONTE: g...,Não encontrei evidências que comprovem esta af...,ADEQUADO,False,[]
2,3,O alumínio é liberado na comida quando aquecid...,[],,INADEQUADO,False,[]
3,4,O governo suspendeu a nova vacina contra dengu...,[ANO: 2026 | FONTE_VERIFICADA: True | FONTE: g...,É perfeitamente compreensível que você sinta a...,ADEQUADO,True,[https://www.gov.br/saude/pt-br/assuntos/saude...
4,5,O nipah tá se espalhando no mundo. Fui no carn...,[ANO: 2026 | FONTE_VERIFICADA: True | FONTE: a...,É perfeitamente compreensível que você sinta a...,ADEQUADO,True,[https://www.aosfatos.org/noticias/o-que-e-vir...


Faça um código python que recebe uma lista nativamente pela leitura de um arquivo json. Cada elemento desta lista é um objeto com chaves e valores. Segue aqui a descrição das chaves e valores, e seu objetivo é colocar cada valor extraído dentro de colunas de um dataframe final.



"id": string

"input_usuario": string

"documentos_retornados_rag": Object com atributos. As chaves são ids e ignore-os, extraia a lista de .values(). Esta lista de strings deve ser o tipo da colunas "documentos_retornados_rag". Processe cada string da lista removendo via regex a substring do início [0] até a substring "ANO:", preservando a parte "ANO:" e eliminado tudo antes.

"resposta_gerada": string

"veredito_validador": string

"fonte_verificada": bool

"referencias": List[str] 

